# 03 HWW EFT fit with repo-consistent response extraction
This notebook intentionally mirrors `stxs_reweight_function` + `stxs_fit` internals but swaps in HWW μ likelihood.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from coffea import util

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p=Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p/marker).exists(): return p
        if p.parent==p: break
        p=p.parent
    raise FileNotFoundError(f'Could not find {marker} from {Path.cwd()}')

REPO=find_repo_root()
NBDIR=REPO/'notebooks_hww_fit'
LISTING=REPO/'smeft_contents.txt'
for p in ['/uscms_data/d3/azhou/smeft/analysis','/uscms_data/d3/azhou/smeft/analysis/hbb-coffea']:
    if p not in sys.path: sys.path.append(p)
import stxs_functions as sf
print('repo:',REPO)
manifest=json.loads((NBDIR/'asset_manifest.json').read_text())
meas=json.loads((NBDIR/'hww_measurement.json').read_text())
coffea_fullpath=manifest['default_coffea_fullpath']
print('using',coffea_fullpath)

In [ ]:
def stxs_coeffs_fullpath(coffea_fullpath, operator_list):
    all_h=util.load(coffea_fullpath)['htxs']
    wc_mapping=sf.wc_map_dict(operator_list)
    b1,b2={},{}
    for p,wcl in wc_mapping.items():
        if wcl not in list(all_h.axes['wc']):
            continue
        y1,y2=sf.get_bin_yields(all_h[{'wc':wcl}])
        b1[p]=y1; b2[p]=y2
    pts=[p for p in wc_mapping if p in b1 and p in b2]
    if len(operator_list)==1:
        x=np.array([p[0] for p in pts]); y1=np.array([b1[p] for p in pts]); y2=np.array([b2[p] for p in pts])
        c1,_=sf.curve_fit(sf.quad_1D,x,y1,p0=[1,1,1]); c2,_=sf.curve_fit(sf.quad_1D,x,y2,p0=[1,1,1])
    elif len(operator_list)==2:
        x1=np.array([p[0] for p in pts]); x2=np.array([p[1] for p in pts]); y1=np.array([b1[p] for p in pts]); y2=np.array([b2[p] for p in pts])
        c1,_=sf.curve_fit(sf.quad_2D,(x1,x2),y1,p0=[1,1,1,1,1,1]); c2,_=sf.curve_fit(sf.quad_2D,(x1,x2),y2,p0=[1,1,1,1,1,1])
    else:
        raise ValueError('Only 1D/2D')
    return c1,c2

In [ ]:
def scan_hww_1d(coffea_fullpath, op, sigma_sm_hww_fb, wc_min=-20,wc_max=20,n=401,mg_sigma_pb=3.594):
    sumw=util.load(coffea_fullpath)['sumw_all_noEW'].value
    c1,c2=stxs_coeffs_fullpath(coffea_fullpath,[op])
    rows=[]
    for x in np.linspace(wc_min,wc_max,n):
        sig1=mg_sigma_pb*1000*sf.quad_1D(x,*c1)/sumw
        sig2=mg_sigma_pb*1000*sf.quad_1D(x,*c2)/sumw
        sig_proxy=sig1+sig2
        mu=sig_proxy/sigma_sm_hww_fb
        d=mu-meas['mu_obs']; s=meas['sig_up'] if d>=0 else meas['sig_dn']
        rows.append({'wc':float(x),'op':op,'sig_proxy_fb':float(sig_proxy),'mu_pred':float(mu),'chi2_hww':float((d/s)**2)})
    return pd.DataFrame(rows)

In [ ]:
def scan_hww_2d(coffea_fullpath, ops, sigma_sm_hww_fb, wc_min=-10,wc_max=10,n=121,mg_sigma_pb=3.594):
    sumw=util.load(coffea_fullpath)['sumw_all_noEW'].value
    c1,c2=stxs_coeffs_fullpath(coffea_fullpath,ops)
    rows=[]
    grid=np.linspace(wc_min,wc_max,n)
    for x in grid:
      for y in grid:
        sig1=mg_sigma_pb*1000*sf.quad_2D((x,y),*c1)/sumw
        sig2=mg_sigma_pb*1000*sf.quad_2D((x,y),*c2)/sumw
        sig_proxy=sig1+sig2
        mu=sig_proxy/sigma_sm_hww_fb
        d=mu-meas['mu_obs']; s=meas['sig_up'] if d>=0 else meas['sig_dn']
        rows.append({'wc1':float(x),'wc2':float(y),'op1':ops[0],'op2':ops[1],'sig_proxy_fb':float(sig_proxy),'mu_pred':float(mu),'chi2_hww':float((d/s)**2)})
    return pd.DataFrame(rows)

In [ ]:
# USER ACTION: set sigma_sm_hww_fb in same fiducial/selection as measurement (m_jj>1TeV).
# If HTXS proxy differs from your fiducial, add transfer factor before interpretation.